#安裝套件
連線記得要切到T4 GPU

In [ ]:
!pip install ultralytics
!pip install albumentations

#連線到雲端硬碟

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

解壓縮你上傳的zip

In [ ]:
# 語法：!unzip "雲端硬碟的壓縮檔路徑" -d "要解壓到的目標資料夾路徑"
import os
target_path = "/content/drive/MyDrive/dataset"
if not os.path.exists(target_path):
  os.makedirs(target_path)
!unzip "/content/drive/MyDrive/car_plate.zip" -d "/content/drive/MyDrive/dataset/car_plate_data"


#引入套件

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Import necessary libraries
import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import cv2
import yaml
import torch
from PIL import Image
from IPython.display import display, Video
from ultralytics import YOLO
from collections import deque
import subprocess  # For executing ffmpeg

#看一下xml跟yaml

In [ ]:
import yaml

# 請確保已掛載雲端硬碟，並填入正確的檔案路徑
yaml_path = '/content/drive/MyDrive/dataset/car_plate_data/data.yaml'

try:
    with open(yaml_path, 'r', encoding='utf-8') as f:
        # 讀取 yaml 檔案內容
        yaml_data = yaml.safe_load(f)

    print("--- data.yaml 讀取成功 ---")
    print(yaml_data)

    # 範例：如果裡面有定義類別名稱（通常 YOLO 格式會寫在 names 欄位）
    if 'names' in yaml_data:
        print("資料集類別名稱：", yaml_data['names'])

except FileNotFoundError:
    print(f"找不到檔案，請檢查路徑是否正確：{yaml_path}")


#看下對應每一個圖片的xml檔案內容，官網說
每張圖片都會對應一個 XML 標註檔案，裡面包含了圖片詳細資訊、邊界框座標、類別、旋轉角度以及其他相關數據。

In [ ]:
import xml.etree.ElementTree as ET

# XML 檔案路徑
xml_path = '/content/drive/MyDrive/dataset/car_plate_data/images/N1.xml'

try:
    # 解析 XML 檔案
    tree = ET.parse(xml_path)
    root = tree.getroot()

    print("--- N1.xml 標註內容 ---")
    # 取得圖片基本資訊
    filename = root.find('filename').text
    width = root.find('size').find('width').text
    height = root.find('size').find('height').text
    print(f"圖片檔名: {filename}, 寬度: {width}, 高度: {height}")

    # 遍歷所有標註的物件（車牌）
    for obj in root.findall('object'):
        label_name = obj.find('name').text
        bndbox = obj.find('bndbox')

        # 抓取車牌邊界框的左上角與右下角座標
        xmin = int(bndbox.find('xmin').text)
        ymin = int(bndbox.find('ymin').text)
        xmax = int(bndbox.find('xmax').text)
        ymax = int(bndbox.find('ymax').text)

        print(f"物件類別: {label_name} -> 邊界框座標: [左上 ({xmin}, {ymin}), 右下 ({xmax}, {ymax})]")

except FileNotFoundError:
    print(f"找不到檔案，請檢查路徑是否正確：{xml_path}")


#畫看看資料跟圖片對不對的起來

In [ ]:
import cv2
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET

# 設定路徑
img_path = '/content/drive/MyDrive/dataset/car_plate_data/images/N1.jpeg'
xml_path = '/content/drive/MyDrive/dataset/car_plate_data/images/N1.xml'

# 1. 讀取圖片 (OpenCV 預設為 BGR，轉為 RGB 供 matplotlib 顯示)
image = cv2.imread(img_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# 2. 解析 XML 取得座標
tree = ET.parse(xml_path)
root = tree.getroot()

for obj in root.findall('object'):
    bndbox = obj.find('bndbox')
    xmin = int(bndbox.find('xmin').text)
    ymin = int(bndbox.find('ymin').text)
    xmax = int(bndbox.find('xmax').text)
    ymax = int(bndbox.find('ymax').text)

    # 3. 在圖片上畫出綠色的矩形框 (框線粗度為 2)
    cv2.rectangle(image, (xmin, ymin), (xmax, ymax), (0, 255, 0), 2)

# 4. 顯示結果
plt.figure(figsize=(10, 6))
plt.imshow(image)
plt.axis('off')
plt.show()


#重新建立新的yaml



In [ ]:
import yaml

yaml_path = '/content/drive/MyDrive/dataset/car_plate_data/data2.yaml'

# 定義正確的路徑與類別（請根據你的實際狀況調整）
# 註：如果你的資料集沒有分 train/val 資料夾，可以先將兩者都指向你的 images 資料夾
data_config = {
    'path': '/content/drive/MyDrive/dataset/car_plate_data', # 資料集根目錄
    'train': 'images', # 訓練集圖片相對路徑 (會自動對應到根目錄下的 images)
    'val': 'images',   # 驗證集圖片相對路徑 (若有 TEST 資料夾也可以改成 'TEST')

    'names': {
        0: 'licence'   # 你的標註類別名稱，例如車牌通常是 licence 或 plate
    }
}

# 覆寫 yaml 檔案
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print("data2.yaml 已成功修正！")


#把資料夾的xml改寫txt並把正確的路徑寫進去

In [ ]:
import os
import xml.etree.ElementTree as ET
import yaml

# ==========================================
# 1. 設定正確的路徑（必須包含 /content/）
# ==========================================
dataset_root = '/content/drive/MyDrive/dataset/car_plate_data'
images_dir = os.path.join(dataset_root, 'images')
yaml_path = os.path.join(dataset_root, 'data2.yaml')

# 類別對應表（車牌通常設為類別 0）
classes = ["licence"]

# ==========================================
# 2. 自動將 XML 轉換為 YOLO TXT 格式
# ==========================================
def convert_voc_to_yolo(xml_file, txt_file):
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()

        # 取得圖片寬高
        size = root.find('size')
        w = int(size.find('width').text)
        h = int(size.find('height').text)

        # 如果寬高為 0，跳過避免除以零錯誤
        if w == 0 or h == 0:
            return False

        with open(txt_file, 'w', encoding='utf-8') as f:
            for obj in root.findall('object'):
                cls_name = obj.find('name').text
                if cls_name not in classes:
                    classes.append(cls_name)
                cls_id = classes.index(cls_name)

                bndbox = obj.find('bndbox')
                xmin = float(bndbox.find('xmin').text)
                ymin = float(bndbox.find('ymin').text)
                xmax = float(bndbox.find('xmax').text)
                ymax = float(bndbox.find('ymax').text)

                # 轉為 YOLO 的相對中心點與寬高 (0~1)
                x_center = (xmin + xmax) / 2.0 / w
                y_center = (ymin + ymax) / 2.0 / h
                bbox_w = (xmax - xmin) / w
                bbox_h = (ymax - ymin) / h

                f.write(f"{cls_id} {x_center:.6f} {y_center:.6f} {bbox_w:.6f} {bbox_h:.6f}\n")
        return True
    except Exception as e:
        print(f"處理 {xml_file} 時發生錯誤: {e}")
        return False

# 開始批量轉換
print("開始轉換標註檔案...")
xml_count = 0
for file in os.listdir(images_dir):
    if file.endswith('.xml'):
        xml_path_file = os.path.join(images_dir, file)
        txt_path_file = os.path.join(images_dir, file.replace('.xml', '.txt'))
        if convert_voc_to_yolo(xml_path_file, txt_path_file):
            xml_count += 1

print(f"成功將 {xml_count} 個 XML 檔案轉換為 YOLO TXT 格式！")


#整理資料夾為標準的檔案架構

In [ ]:
import os
import shutil
import xml.etree.ElementTree as ET
import yaml

# ==========================================
# 1. 定義路徑
# ==========================================
base_path = '/content/drive/MyDrive/dataset/car_plate_data'

# 舊的資料夾路徑
old_images_dir = os.path.join(base_path, 'images')
old_test_dir = os.path.join(base_path, 'TEST')

# 新的標準 YOLO 結構路徑
new_train_img = os.path.join(base_path, 'images_new/train')
new_val_img = os.path.join(base_path, 'images_new/val')
new_train_txt = os.path.join(base_path, 'labels/train')
new_val_txt = os.path.join(base_path, 'labels/val')

# 建立所有新資料夾
os.makedirs(new_train_img, exist_ok=True)
os.makedirs(new_val_img, exist_ok=True)
os.makedirs(new_train_txt, exist_ok=True)
os.makedirs(new_val_txt, exist_ok=True)

classes = ["license_plate"]

# ==========================================
# 2. XML 轉 YOLO TXT 函數
# ==========================================
def convert_xml_to_txt(xml_path, output_txt_path):
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        size = root.find('size')
        w = int(size.find('width').text)
        h = int(size.find('height').text)
        if w == 0 or h == 0: return False

        with open(output_txt_path, 'w', encoding='utf-8') as f:
            for obj in root.findall('object'):
                bndbox = obj.find('bndbox')
                xmin = float(bndbox.find('xmin').text)
                ymin = float(bndbox.find('ymin').text)
                xmax = float(bndbox.find('xmax').text)
                ymax = float(bndbox.find('ymax').text)

                # 轉為 YOLO 相對座標
                x_center = (xmin + xmax) / 2.0 / w
                y_center = (ymin + ymax) / 2.0 / h
                bbox_w = (xmax - xmin) / w
                bbox_h = (ymax - ymin) / h

                f.write(f"0 {x_center:.6f} {y_center:.6f} {bbox_w:.6f} {bbox_h:.6f}\n")
        return True
    except:
        return False

# ==========================================
# 3. 開始搬移與轉換資料
# ==========================================
def process_folder(src_dir, dest_img_dir, dest_txt_dir):
    img_count, txt_count = 0, 0
    if not os.path.exists(src_dir): return 0, 0

    for file in os.listdir(src_dir):
        file_path = os.path.join(src_dir, file)
        # 處理圖片
        if file.lower().endswith(('.jpeg', '.jpg', '.png')):
            shutil.copy(file_path, os.path.join(dest_img_dir, file))
            img_count += 1

            # 尋找同名的 XML 進行轉換
            xml_name = os.path.splitext(file)[0] + '.xml'
            xml_path = os.path.join(src_dir, xml_name)
            if os.path.exists(xml_path):
                txt_name = os.path.splitext(file)[0] + '.txt'
                if convert_xml_to_txt(xml_path, os.path.join(dest_txt_dir, txt_name)):
                    txt_count += 1
    return img_count, txt_count

print("正在處理訓練集（原本的 images 資料夾）...")
train_img, train_txt = process_folder(old_images_dir, new_train_img, new_train_txt)
print(f"-> 訓練集完成：複製了 {train_img} 張圖片，轉換了 {train_txt} 個標註檔。")

print("\n正在處理驗證集（原本的 TEST 資料夾）...")
val_img, val_txt = process_folder(old_test_dir, new_val_img, new_val_txt)
print(f"-> 驗證集完成：複製了 {val_img} 張圖片，轉換了 {val_txt} 個標註檔。")

# ==========================================
# 4. 把臨時的 images_new 取代原本的 images
# ==========================================
# 先移除舊的圖片資料夾，再把整理好的換過去
try:
    shutil.rmtree(old_images_dir)
    if os.path.exists(old_test_dir): shutil.rmtree(old_test_dir)
    shutil.move(os.path.join(base_path, 'images_new'), old_images_dir)
    print("\n資料夾結構更換成功！已轉為標準 YOLO 格式。")
except Exception as e:
    print(f"\n更換資料夾時發生小錯誤（通常是雲端硬碟同步延遲）：{e}")
    print("請手動重新整理左側檔案樹檢查。")

# ==========================================
# 5. 更新 data.yaml
# ==========================================
yaml_path = os.path.join(base_path, 'data2.yaml')
data_config = {
    'path': base_path,
    'train': 'images/train',
    'val': 'images/train',
    'nc': 1,
    'names': {0: 'license_plate'}
}
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(data_config, f, default_flow_style=False)
print("data.yaml 已成功重寫為標準相對路徑！")


#準備訓練模型

In [ ]:
model = YOLO("yolov8n.pt")

results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    lr0= 0.0005,
    batch=32,
    lrf=0.1,
    augment=True,
    # 加上這行，結果會直接同步寫入你的雲端硬碟！
    project='/content/drive/MyDrive/dataset/car_plate_data/runs'
)



#訓練結果

In [ ]:
# train_results = "/content/runs/detect/train/results.png"

train_results = '/content/drive/MyDrive/dataset/car_plate_data/runs/train/results.png'

if os.path.exists(train_results):
    img = Image.open(train_results)  # Use PIL to read the image
    display(img)  # Display the image
else:
    print("Training results image not found.")

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

# Colab 的標準 YOLOv8 訓練結果路徑
# train_results = "/content/runs/detect/train/results.png"
train_results = '/content/drive/MyDrive/dataset/car_plate_data/runs/train/results.png'



if os.path.exists(train_results):
    img = Image.open(train_results)

    # 在 Colab 中使用 matplotlib 顯示圖片會最穩定漂亮
    plt.figure(figsize=(12, 8))
    plt.imshow(img)
    plt.axis('off') # 隱藏坐標軸
    plt.show()
else:
    print(f"找不到訓練結果圖片，請檢查路徑是否存在：{train_results}")
    print("提示：請確認左側檔案樹的 runs/detect/ 資料夾下的名稱是 train 還是 train2？")


#如果要寫到雲端

In [ ]:
results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    lr0=0.0005,
    batch=32,
    lrf=0.1,
    augment=True,
    # 加上這行，結果會直接同步寫入你的雲端硬碟！
    project='/content/drive/MyDrive/dataset/car_plate_data/runs'
)

In [ ]:
!cp -r "/content/runs" "/content/drive/MyDrive/dataset/car_plate_data/"
# 需要等一下


In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

# Colab 的標準 YOLOv8 訓練結果路徑
# train_results = '/content/drive/MyDrive/dataset/car_plate_data/runs/detect/train/results.png'
train_results = '/content/drive/MyDrive/dataset/car_plate_data/runs/train/results.png'


if os.path.exists(train_results):
    img = Image.open(train_results)

    # 在 Colab 中使用 matplotlib 顯示圖片會最穩定漂亮
    plt.figure(figsize=(12, 8))
    plt.imshow(img)
    plt.axis('off') # 隱藏坐標軸
    plt.show()
else:
    print(f"找不到訓練結果圖片，請檢查路徑是否存在：{train_results}")
    print("提示：請確認左側檔案樹的 runs/detect/ 資料夾下的名稱是 train 還是 train2？")

# 隨機測試幾張圖

In [ ]:
import cv2
import glob
import random
import os
import matplotlib.pyplot as plt
from ultralytics import YOLO

# ==========================================
# 1. 設定雲端硬碟中的模型與圖片路徑
# ==========================================
# 載入你剛剛備份到雲端硬碟的最佳模型權重
model_path = '/content/drive/MyDrive/dataset/car_plate_data/runs/train/weights/best.pt'
model = YOLO(model_path)

# 蒐集雲端硬碟資料夾內所有的 .jpeg 圖片 (這裡改為你的訓練集圖片路徑)
image_folder = "/content/drive/MyDrive/dataset/car_plate_data/images/train/*.jpeg"
all_images = glob.glob(image_folder)

# 安全檢查：確認有沒有抓到圖片
if len(all_images) == 0:
    print(f"❌ 錯誤：在路徑下找不到任何 .jpeg 圖片，請檢查資料夾路徑是否正確：{image_folder}")
else:
    # ==========================================
    # 2. 隨機抽選 6 張不同的圖片
    # ==========================================
    sample_size = min(6, len(all_images)) # 如果圖片少於6張，就全部分析
    test_images = random.sample(all_images, sample_size)
    print(f"🎉 成功找到 {len(all_images)} 張圖片，隨機抽出 {sample_size} 張進行測試...")

    # ==========================================
    # 3. 建立 2 欄 3 列的畫布並進行預測
    # ==========================================
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))

    # 使用 flatten() 把 2x3 的矩陣變成一維陣列，方便用對應的方式跑迴圈
    for img_path, ax in zip(test_images, axes.flatten()):
        results = model(img_path)      # 讓 YOLOv8 模型進行車牌預測
        result_img = results[0].plot() # 取得自動畫好綠色框與標籤的圖片結果

        # 將 OpenCV 預設的 BGR 圖片轉換為 Matplotlib 支援的 RGB 格式
        img_rgb = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)

        ax.imshow(img_rgb)
        ax.axis("off") # 隱藏坐標軸讓畫面乾淨
        ax.set_title(os.path.basename(img_path))  # 將圖片檔名顯示為標題

    # 如果抽出來的圖片不滿 6 張，把多餘的空白格子隱藏起來
    if sample_size < 6:
        for i in range(sample_size, 6):
            axes.flatten()[i].axis('off')

    plt.tight_layout()
    plt.show()


#用影片來測試

In [ ]:
from ultralytics import YOLO
import cv2
import os
import subprocess
from IPython.display import HTML
from base64 import b64encode

# ==========================================
# 1. 設定 Google Colab 中的正確路徑
# ==========================================
# 載入你之前備份到雲端硬碟的最佳權重檔案
model_path = "/content/drive/MyDrive/dataset/car_plate_data/runs/train/weights/best.pt"
model = YOLO(model_path)

# 🚨 請確保你的雲端硬碟有這個影片，或者將路徑改成你上傳的影片路徑
input_video = "/content/drive/MyDrive/dataset/car_plate_data/car_video.mp4"

# Colab 的輸出檔案路徑（先存在 Colab 本機環境，速度較快）
output_video = "/content/output_video.mp4"
compressed_video = "/content/output_video_compressed.mp4"

# 安全檢查：確認輸入影片是否存在
if not os.path.exists(input_video):
    print(f"❌ 錯誤：找不到輸入影片檔案！請檢查路徑：{input_video}")
else:
    print("🎬 開始讀取影片並進行車牌偵測...")

    # 開啟影片檔案
    cap = cv2.VideoCapture(input_video)

    # 取得影片屬性
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # 設定影片編碼器與輸出目標
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

    # ==========================================
    # 2. 逐幀（Frame by Frame）進行 YOLOv8 偵測
    # ==========================================
    frame_count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # 執行 YOLOv8 預測 (verbose=False 關閉逐幀的 log 輸出，畫面才乾淨)
        results = model(frame, verbose=False)

        # 繪製車牌偵測的綠色框與文字
        for result in results:
            for box in result.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf = box.conf[0].item()
                label = f"Plate {conf:.2f}"

                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        # 寫入影格到輸出影片中
        out.write(frame)
        frame_count += 1

        # 每 100 幀印出一點進度，讓你知道程式有在跑
        if frame_count % 100 == 0:
            print(f"⏳ 已處理 {frame_count} 幀...")

    # 釋放記憶體與影片資源
    cap.release()
    out.release()
    # cv2.destroyAllWindows()
    print(f"✅ 偵測完成！總共處理了 {frame_count} 幀。")

    # ==========================================
    # 3. 使用 FFmpeg 壓縮影片（網頁瀏覽器才播得出來）
    # ==========================================
    print("⏳ 正在壓縮影片格式以相容瀏覽器播放... (可能需要數十秒)")
    ffmpeg_command = [
        "ffmpeg", "-y", "-i", output_video, "-vcodec", "libx264", "-crf", "28", "-preset", "fast", compressed_video
    ]
    # 執行 Linux 壓縮指令
    res = subprocess.run(ffmpeg_command, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

    if res.returncode == 0:
        print("✅ 影片壓縮成功！")

        # 可選：把壓縮好的影片也備份一份到雲端硬碟
        shutil.copy(compressed_video, "/content/drive/MyDrive/dataset/car_plate_data/output_video_compressed.mp4")
    else:
        print("❌ FFmpeg 壓縮失敗，錯誤訊息：", res.stderr.decode())

# ==========================================
# 4. 定義在網頁內直接播放影片的函數
# ==========================================
def play_video(file_path, width=800):
    """將影片轉為 Base64 碼並直接在 Colab 網頁內嵌播放"""
    if not os.path.exists(file_path):
        return HTML("<p style='color:red;'>🚨 錯誤：找不到壓縮後的影片檔案！</p>")

    with open(file_path, "rb") as video_file:
        video_data = video_file.read()
        video_base64 = b64encode(video_data).decode()

    video_html = f"""
    <video width="{width}" controls autoplay loop>
        <source src="data:video/mp4;base64,{video_base64}" type="video/mp4">
        您的瀏覽器不支援 HTML5 影片標籤。
    </video>
    """
    return HTML(video_html)

# 顯示最後結果
play_video(compressed_video, width=1000)
